## Hypotheses and Test Plan

**Question:** Should the shop switch from the old landing page to the new one?

**Null hypothesis (H0):** The old page and the new page have the same conversion rate. Any difference we see in the data is just random chance.

**Alternative hypothesis (H1):** The old page and the new page have different conversion rates.

**Test I will use:** Chi-square test of independence (`scipy.stats.chi2_contingency`) on a 2x2 table of counts (page vs. converted yes/no).

**Why this test:** Each visitor either bought (1) or did not buy (0), so I am comparing two proportions from two separate groups of visitors. Chi-square compares the counts I actually observed with the counts I would expect if the page made no difference.

**Significance level (alpha):** 0.05. If the p value is below 0.05, I reject H0. This means I accept a 5% risk of a false alarm (saying the page matters when it does not).

In [12]:

import pandas as pd

url = "https://raw.githubusercontent.com/Suryaday/Analyze-AB-Test-Results-Udacity-Data-Analyst-Nanodegree/master/ab_data.csv"
df = pd.read_csv(url)

print(df.shape)
df.head()

(294478, 5)


,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [13]:
mismatch = (
    ((df["group"] == "control") & (df["landing_page"] == "new_page")) |
    ((df["group"] == "treatment") & (df["landing_page"] == "old_page"))
)

print("Mismatched rows:", mismatch.sum())

df2 = df[~mismatch].copy()
print("Rows after removal:", len(df2))

Mismatched rows: 3893
Rows after removal: 290585


In [14]:
print("Duplicate user_ids:", df2["user_id"].duplicated().sum())

df2 = df2.drop_duplicates(subset="user_id", keep="first")

print("Rows now:", len(df2))
print("Unique users:", df2["user_id"].nunique())

Duplicate user_ids: 1
Rows now: 290584
Unique users: 290584


In [15]:
old_rate = df2[df2["landing_page"] == "old_page"]["converted"].mean()
new_rate = df2[df2["landing_page"] == "new_page"]["converted"].mean()

diff_pp = (new_rate - old_rate) * 100

print(f"Old page conversion rate: {old_rate:.4f} ({old_rate*100:.2f}%)")
print(f"New page conversion rate: {new_rate:.4f} ({new_rate*100:.2f}%)")
print(f"Difference (new - old): {diff_pp:.2f} percentage points")

Old page conversion rate: 0.1204 (12.04%)
New page conversion rate: 0.1188 (11.88%)
Difference (new - old): -0.16 percentage points


In [16]:
table = pd.crosstab(df2["landing_page"], df2["converted"])
print(table)

converted          0      1
landing_page               
new_page      128046  17264
old_page      127785  17489


In [17]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(table)

print("Chi-square statistic:", round(chi2, 3))
print("p value:", round(p_value, 4))
print("Degrees of freedom:", dof)
print("Expected counts if the pages were equal:")
print(expected.round(0))


Chi-square statistic: 1.704
p value: 0.1918
Degrees of freedom: 1
Expected counts if the pages were equal:
[[127931.  17379.]
 [127900.  17374.]]


In [18]:
alpha = 0.05

if p_value < alpha:
    print("Reject H0: the difference is statistically significant.")
else:
    print("Fail to reject H0: not enough evidence of a real difference.")
    

Fail to reject H0: not enough evidence of a real difference.


In [19]:
import numpy as np

n_old = (df2["landing_page"] == "old_page").sum()
n_new = (df2["landing_page"] == "new_page").sum()

print("Visitors on old page:", n_old)
print("Visitors on new page:", n_new)

Visitors on old page: 145274
Visitors on new page: 145310


In [20]:
diff = new_rate - old_rate

se = np.sqrt(
    old_rate * (1 - old_rate) / n_old +
    new_rate * (1 - new_rate) / n_new
)

lower = diff - 1.96 * se
upper = diff + 1.96 * se

print(f"Difference: {diff*100:.2f} percentage points")
print(f"Standard error: {se*100:.3f} percentage points")
print(f"95% CI: [{lower*100:.2f}, {upper*100:.2f}] percentage points")

Difference: -0.16 percentage points
Standard error: 0.120 percentage points
95% CI: [-0.39, 0.08] percentage points


In [21]:
if lower < 0 < upper:
    print("The interval includes 0: the true difference could be zero.")
else:
    print("The interval does not include 0: the difference looks real.")

The interval includes 0: the true difference could be zero.


## What the p value means

**What it means here:** My p value is about 0.19. This means that if the old and new pages truly had the same conversion rate, I would still see a difference as big as the one in my data (or bigger) about 19% of the time, just by random chance. That is not unusual enough to call the difference real, so I fail to reject the null hypothesis.

**What it does NOT mean:** It does not mean there is a 19% chance that the null hypothesis is true, and it does not prove that the two pages are equal. It only says my data is not surprising enough to rule out luck. A high p value means "not enough evidence", not "proof of no difference".

## Verdict for the founder

**Decision: Keep the old page. Do not switch.**

1. The new page did not perform better: it converted at 11.88% versus 12.04% for the old page, and the small gap is within what chance alone can produce (p = 0.19).
2. We tested about 290,000 visitors, and the plausible true effect of the new page ranges from -0.39 to +0.08 percentage points, so even the best case is a tiny gain and the likely case is no change or a small loss.
3. Switching would cost effort with no evidence of benefit, so stay with the old page and test a new design idea instead.